In [1]:
#Defining the storage paths which includes the storage account name and container name
storage_account_name = "nyctaxi100"
container_name = "taxidata"

raw_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/raw/"
processed_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/processed/"
curated_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/curated/"
#Printing and displaying a confirmation message
print("Paths have been configured!")

In [2]:
#Importing all the necessary libraries
import pyspark.sql.functions as F
from pyspark.sql.types import *

#Disabling the vectorized reading
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")
spark.conf.set("spark.sql.legacy.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.legacy.parquet.datetimeRebaseModeInRead", "CORRECTED")

#Listing all 12 months of nyc taxi data in 2023
months = [
    "2023-01", "2023-02", "2023-03", "2023-04",
    "2023-05", "2023-06", "2023-07", "2023-08",
    "2023-09", "2023-10", "2023-11", "2023-12"
]

#Printing and displaying the validation report for all 12 months in 2023
print("=" * 50)
print("VALIDATION REPORT OF NYC TAXI IN 2023")
print("=" * 50)

total_raw = 0
total_curated = 0

for m in months:
    try:
        #Reading the raw NYC taxi file
        raw_file = f"{raw_path}yellow_tripdata_{m}.parquet"
        df_raw = spark.read \
            .option("inferSchema", "false") \
            .parquet(raw_file)
        raw_count = df_raw.count()
        total_raw += raw_count

        #Reading the curated NYC taxi file
        curated_file = f"{curated_path}yellow_tripdata_{m}.parquet"
        df_curated = spark.read \
            .option("inferSchema", "false") \
            .parquet(curated_file)
        curated_count = df_curated.count()
        total_curated += curated_count

        #Calculating the number of rows removed
        removed = raw_count - curated_count
        retention = int((curated_count / raw_count) * 10000) / 100
        #Printing and displaying the rows for raw, curated, removed and retention
        print(f"\n{m}:")
        print(f"  Raw rows:     {raw_count:,}")
        print(f"  Curated rows: {curated_count:,}")
        print(f"  Removed:      {removed:,}")
        print(f"  Retention:    {retention}%")

    except Exception as e:
        print(f"❌ {m} failed: {str(e)[:200]}")

#Calculating the overall retention without rounding up or down
overall_retention = int((total_curated / total_raw) * 10000) / 100
#Printing and displaying the relevant information regarding the nyc taxi validation report
print("\n" + "=" * 50)
print(f"TOTAL RAW ROWS:     {total_raw:,}")
print(f"TOTAL CURATED ROWS: {total_curated:,}")
print(f"TOTAL REMOVED:      {total_raw - total_curated:,}")
print(f"OVERALL RETENTION:  {overall_retention}%")
print("=" * 50)